# 🌍 NASA Space Apps Challenge: Be An Earth System Trend Detective!
## Phase 1: Long-Term Surface Air Temperature (T2M) Trend & Statistical Significance Analysis

**Investigative Target:** Dhaka, Bangladesh (Lat: 23.8103°N, Lon: 90.4125°E)  
**Primary Variable:** Air Temperature at 2 Meters (`T2M`, °C)  
**Data Source:** NASA Goddard Modeling and Assimilation Office (GMAO) MERRA-2 via NASA POWER API  
**Temporal Window:** 2001-01-01 to 2025-12-31 (25 Years Daily Resolution, 9,131 Observations)  
**Project:** Orion Space - Earth System Intelligence Engine

---
### 🎯 The Detective Question:
> *"How has surface air temperature changed over Bangladesh over the multi-decadal long term, and is the observed trend a scientifically verified change or an artifact of statistical noise?"*


In [1]:
import os
import sys
import requests
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt

# Set plot styling for scientific publication standards
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.size'] = 11
plt.rcParams['figure.dpi'] = 120

print("All scientific libraries imported successfully.")

All scientific libraries imported successfully.


---
## Step A — Ingesting NASA POWER / MERRA-2 Climate Data

NASA's Prediction Of Worldwide Energy Resources (POWER) project synthesizes NASA satellite observations and GMAO MERRA-2 assimilation model outputs. We query the daily point endpoint for 25 complete years (2001–2025).

In [2]:
NASA_POWER_URL = "https://power.larc.nasa.gov/api/temporal/daily/point"
CACHE_PATH = "../data/dhaka_t2m_2001_2025.csv"

# Parameters
LATITUDE = 23.8103
LONGITUDE = 90.4125
START_DATE = "20010101"
END_DATE = "20251231"

if os.path.exists(CACHE_PATH):
    print(f"[CACHE] Loading local cached dataset from: {CACHE_PATH}")
    df_daily = pd.read_csv(CACHE_PATH, index_col="date", parse_dates=True)
else:
    print(f"[API] Fetching from NASA POWER API for Lat: {LATITUDE}, Lon: {LONGITUDE}...")
    params = {
        "parameters": "T2M",
        "community": "RE",
        "longitude": LONGITUDE,
        "latitude": LATITUDE,
        "start": START_DATE,
        "end": END_DATE,
        "format": "JSON"
    }
    response = requests.get(NASA_POWER_URL, params=params, timeout=60)
    response.raise_for_status()
    data = response.json()
    
    # Extract T2M time series
    t2m_dict = data["properties"]["parameter"]["T2M"]
    df_daily = pd.DataFrame(list(t2m_dict.items()), columns=["date", "T2M"])
    df_daily["date"] = pd.to_datetime(df_daily["date"], format="%Y%m%d")
    df_daily.set_index("date", inplace=True)
    
    # Handle NASA fill value (-999.0)
    df_daily.replace(-999.0, np.nan, inplace=True)
    df_daily["T2M"] = pd.to_numeric(df_daily["T2M"], errors="coerce")
    
    os.makedirs(os.path.dirname(CACHE_PATH), exist_ok=True)
    df_daily.to_csv(CACHE_PATH)
    print(f"[SAVED] Saved {len(df_daily)} daily records to {CACHE_PATH}")

print(f"Dataset ready. Total observations: {len(df_daily):,}")

[CACHE] Loading local cached dataset from: ../data/dhaka_t2m_2001_2025.csv
Dataset ready. Total observations: 9,131


---
## Step B — Data Quality Audit & Understanding

A true detective inspects the evidence before drawing conclusions:
1. **Date Bounds:** Check start and end dates
2. **Missing Values:** Ensure missing / fill values are identified
3. **Duplicates:** Confirm strict daily continuity
4. **Physical Sanity Check:** Verify temperatures are within realistic subtropical meteorological ranges (°C)

In [3]:
print("=== DATA QUALITY AUDIT ===")
print(f"Start Date:            {df_daily.index.min().strftime('%Y-%m-%d')}")
print(f"End Date:              {df_daily.index.max().strftime('%Y-%m-%d')}")
print(f"Total Observations:    {len(df_daily):,} days")
print(f"Missing Values (NaN):  {df_daily['T2M'].isna().sum()}")
print(f"Duplicate Dates:       {df_daily.index.duplicated().sum()}")
print(f"Minimum Daily T2M:     {df_daily['T2M'].min():.2f} °C")
print(f"Maximum Daily T2M:     {df_daily['T2M'].max():.2f} °C")
print(f"Mean Daily T2M:        {df_daily['T2M'].mean():.2f} °C")
print(f"Standard Deviation:    {df_daily['T2M'].std():.2f} °C")

df_daily.head(10)

=== DATA QUALITY AUDIT ===
Start Date:            2001-01-01
End Date:              2025-12-31
Total Observations:    9,131 days
Missing Values (NaN):  0
Duplicate Dates:       0
Minimum Daily T2M:     10.71 °C
Maximum Daily T2M:     34.51 °C
Mean Daily T2M:        25.51 °C
Standard Deviation:    4.40 °C


---
## Step C — Multi-Scale Temporal Aggregation (Daily → Monthly → Yearly)

Daily climate measurements exhibit high stochastic noise (synoptic weather fluctuations, storm events). To detect multi-decadal trends, we aggregate daily records into:
- **Monthly Mean ($T_{monthly}$):** Captures seasonal cycle dynamics (Winter, Pre-monsoon summer, Monsoon, Post-monsoon)
- **Annual Mean ($T_{yearly}$):** Isolates the multi-decadal climate signal

In [4]:
# Monthly Aggregation
df_monthly = df_daily[['T2M']].resample('ME').mean()
df_monthly.rename(columns={'T2M': 'T2M_monthly'}, inplace=True)

# Annual Aggregation
df_yearly = df_daily[['T2M']].resample('YE').mean()
df_yearly.rename(columns={'T2M': 'T2M_yearly'}, inplace=True)
df_yearly['year'] = df_yearly.index.year

fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=False)

# 1. Daily Raw Data
axes[0].plot(df_daily.index, df_daily['T2M'], color='#6baed6', alpha=0.5, linewidth=0.6, label='Daily T2M (Weather Noise)')
axes[0].set_title('Level 1: Daily Raw Measurements (High Synoptic Variability)', fontweight='bold')
axes[0].set_ylabel('T2M (°C)')
axes[0].legend(loc='upper right')

# 2. Monthly Mean
axes[1].plot(df_monthly.index, df_monthly['T2M_monthly'], color='#fd8d3c', linewidth=1.2, label='Monthly Mean (Seasonal Cycle)')
axes[1].set_title('Level 2: Monthly Mean (Seasonal Cycles Revealed)', fontweight='bold')
axes[1].set_ylabel('T2M (°C)')
axes[1].legend(loc='upper right')

# 3. Annual Mean
axes[2].plot(df_yearly['year'], df_yearly['T2M_yearly'], marker='o', color='#d94801', linewidth=2.0, label='Annual Mean (Climate Signal)')
axes[2].set_title('Level 3: Annual Mean Time Series (Decadal Signal)', fontweight='bold')
axes[2].set_xlabel('Year')
axes[2].set_ylabel('T2M (°C)')
axes[2].legend(loc='upper right')

plt.tight_layout()
plt.show()

---
## Step D & E — Trend Quantification & Statistical Significance Testing

Here is the scientific cornerstone of the challenge:  
We calculate:
1. **Linear Trend Slope ($m$):** Ordinary Least Squares (OLS) rate of change per year and per decade
2. **Total Change ($\Delta T$):** Expected total temperature shift over the 25-year baseline
3. **Percentage Change ($\% \Delta T$):** Relative magnitude of variation
4. **OLS p-value:** Null hypothesis test ($H_0: \text{slope} = 0$)
5. **Mann-Kendall Trend Test ($S, Z, p$):** Non-parametric rank-based test robust against non-normality
6. **Sen's Slope Estimator:** Median of all pairwise slopes, impervious to extreme climate anomalies
7. **Verdict:** Is $p < 0.05$ (Statistically Significant) or is it a "Statistical Illusion"?

In [5]:
years = df_yearly['year'].values
annual_temps = df_yearly['T2M_yearly'].values

# 1. Parametric Ordinary Least Squares (OLS) Regression
ols = stats.linregress(years, annual_temps)
slope_ols = ols.slope
intercept_ols = ols.intercept
r_sq = ols.rvalue ** 2
p_val_ols = ols.pvalue
trend_line_ols = intercept_ols + slope_ols * years
total_change_ols = trend_line_ols[-1] - trend_line_ols[0]
pct_change_ols = (total_change_ols / abs(trend_line_ols[0])) * 100

# 2. Non-Parametric Mann-Kendall Test & Sen's Slope
n = len(annual_temps)
s = 0
pairwise_slopes = []
for i in range(n - 1):
    for j in range(i + 1, n):
        diff = annual_temps[j] - annual_temps[i]
        s += np.sign(diff)
        pairwise_slopes.append(diff / (j - i))

sens_slope = float(np.median(pairwise_slopes))

# Variance of S
var_s = (n * (n - 1) * (2 * n + 5)) / 18.0
if s > 0:
    z_mk = (s - 1) / np.sqrt(var_s)
elif s < 0:
    z_mk = (s + 1) / np.sqrt(var_s)
else:
    z_mk = 0.0
p_val_mk = 2.0 * (1.0 - stats.norm.cdf(abs(z_mk)))

# Format End Goal Scientific Table
print("=" * 75)
print("           ORION SPACE - CLIMATE TREND DETECTIVE RESULT TABLE")
print("=" * 75)
print(f"Location:                    Dhaka, Bangladesh (23.8103°N, 90.4125°E)")
print(f"Variable:                    Air Temperature at 2 m (T2M) [NASA MERRA-2]")
print(f"Period:                      2001 – 2025 (25 Years)")
print("-" * 75)
print(f"Trend Direction:             {'Increasing' if slope_ols > 0 else 'Decreasing'}")
print(f"OLS Slope:                   {slope_ols:+.4f} °C / year  ({slope_ols*10:+.3f} °C / decade)")
print(f"Sen's Slope:                 {sens_slope:+.4f} °C / year  ({sens_slope*10:+.3f} °C / decade)")
print(f"Total Change (2001-2025):    {total_change_ols:+.2f} °C")
print(f"Percentage Change:           {pct_change_ols:+.2f} %")
print(f"Coefficient of Det (R²):     {r_sq:.4f}")
print(f"Linear Regression p-value:   {p_val_ols:.4f}")
print(f"Mann-Kendall p-value:        {p_val_mk:.4f} (Z = {z_mk:+.2f})")
print("-" * 75)
sig_verdict = "YES (Statistically Significant)" if (p_val_ols < 0.05 or p_val_mk < 0.05) else "NO (Statistical Illusion / Not Significant at α=0.05)"
print(f"Statistically Significant:   {sig_verdict}")
print("=" * 75)

           ORION SPACE - CLIMATE TREND DETECTIVE RESULT TABLE
Location:                    Dhaka, Bangladesh (23.8103°N, 90.4125°E)
Variable:                    Air Temperature at 2 m (T2M) [NASA MERRA-2]
Period:                      2001 – 2025 (25 Years)
---------------------------------------------------------------------------
Trend Direction:             Increasing
OLS Slope:                   +0.0026 °C / year  (+0.026 °C / decade)
Sen's Slope:                 +0.0036 °C / year  (+0.036 °C / decade)
Total Change (2001-2025):    +0.06 °C
Percentage Change:           +0.24 %
Coefficient of Det (R²):     0.0042
Linear Regression p-value:   0.7571
Mann-Kendall p-value:        0.6913 (Z = +0.40)
---------------------------------------------------------------------------
Statistically Significant:   NO (Statistical Illusion / Not Significant at α=0.05)


---
## Step F — Definitive Publication-Quality Visual Evidence

We now generate the definitive investigative visualization featuring:
- Observed Annual Mean Temperature Points with connecting trajectory
- Ordinary Least Squares (OLS) Linear Trend with 95% Confidence Interval band
- Theil-Sen Robust Estimator Line
- Climatological Detective Diagnostic callout box

In [6]:
plt.figure(figsize=(12, 7))

# Calculate 95% confidence interval for regression line
x_vals = years
y_vals = annual_temps
n_obs = len(x_vals)
x_mean = np.mean(x_vals)
se_line = ols.stderr * np.sqrt(1/n_obs + (x_vals - x_mean)**2 / np.sum((x_vals - x_mean)**2))
t_crit = stats.t.ppf(0.975, df=n_obs-2)
ci_upper = trend_line_ols + t_crit * se_line
ci_lower = trend_line_ols - t_crit * se_line

# Plot observations
plt.plot(x_vals, y_vals, marker='o', color='#1f77b4', markersize=7, linewidth=2.0, label='Annual Mean T2M (NASA MERRA-2)', zorder=4)

# Plot OLS regression line
plt.plot(x_vals, trend_line_ols, color='#d62728', linestyle='--', linewidth=2.2, 
         label=f'OLS Trend ({slope_ols*10:+.2f}°C/decade, p={p_val_ols:.3f})', zorder=5)

# Plot Sen's slope
sen_line = np.median(y_vals) + sens_slope * (x_vals - np.median(x_vals))
plt.plot(x_vals, sen_line, color='#2ca02c', linestyle=':', linewidth=2.0, 
         label=f'Sen\'s Slope ({sens_slope*10:+.2f}°C/decade, p={p_val_mk:.3f})', zorder=5)

# Confidence Band
plt.fill_between(x_vals, ci_lower, ci_upper, color='#d62728', alpha=0.15, label='95% Confidence Interval')

# Labels and Title
plt.title('NASA POWER / MERRA-2 Climatological Trend Analysis (2001–2025)\nDhaka, Bangladesh — Surface Air Temperature at 2m (T2M)', 
          fontsize=14, fontweight='bold', pad=15)
plt.xlabel('Year', fontsize=12, fontweight='bold')
plt.ylabel('Annual Mean Temperature (°C)', fontsize=12, fontweight='bold')
plt.xticks(np.arange(2001, 2026, 2))
plt.ylim(y_vals.min() - 0.5, y_vals.max() + 0.8)

# Diagnostic Text Box
textbox_content = (
    f"DETECTIVE VERDICT:\n"
    f"-----------------------------\n"
    f"Slope (OLS):   {slope_ols:+.4f} °C/yr\n"
    f"Total Change:  {total_change_ols:+.2f} °C\n"
    f"OLS p-value:   {p_val_ols:.3f}\n"
    f"M-K p-value:   {p_val_mk:.3f}\n"
    f"Significance:  Not Sig. (p > 0.05)\n"
    f"Classification: Annual Mean Buffer"
)
plt.text(0.02, 0.95, textbox_content, transform=plt.gca().transAxes, fontsize=10,
         verticalalignment='top', bbox=dict(boxstyle='round,pad=0.6', facecolor='#f8f9fa', edgecolor='#adb5bd', alpha=0.9))

plt.legend(loc='lower right', frameon=True, framealpha=0.9)
plt.tight_layout()
plt.savefig('../data/dhaka_t2m_trend_analysis.png', dpi=300)
plt.show()

---
## Step G — The Detective's Deeper Climatological Discovery: Why is Annual Mean Not Significant?

### The Detective's Paradox:
Why is the overall annual mean warming (+0.06°C over 25 years) yielding $p = 0.757$ (not statistically significant), despite widespread reports of extreme heat events in Bangladesh?

Let us test the **Seasonal Deconstruction Hypothesis**:
In Bangladesh's subtropical monsoon climate:
1. **Monsoon Buffering:** Heavy monsoon rains and cloud cover during June–September keep summer peak temperatures suppressed.
2. **Winter & Pre-Monsoon Divergence:** Are specific months (e.g., April heatwaves or Winter minimums) warming with high statistical significance while being masked by the annual average?

Let us inspect the trend for each month separately.

In [7]:
df_daily['month'] = df_daily.index.month
df_daily['year'] = df_daily.index.year

monthly_summary = []
for m in range(1, 13):
    month_data = df_daily[df_daily['month'] == m].groupby('year')['T2M'].mean()
    m_years = month_data.index.values
    m_temps = month_data.values
    m_ols = stats.linregress(m_years, m_temps)
    monthly_summary.append({
        'Month': pd.to_datetime(f'2020-{m:02d}-01').strftime('%B'),
        'Slope (°C/decade)': round(m_ols.slope * 10, 3),
        'Total Change (°C)': round(m_ols.slope * 25, 2),
        'p-value': round(m_ols.pvalue, 4),
        'Significant (p<0.05)': 'YES' if m_ols.pvalue < 0.05 else 'NO'
    })

df_month_trends = pd.DataFrame(monthly_summary)
print("=== SEASONAL BREAKDOWN OF TEMPERATURE TRENDS (DHAKA, 2001-2025) ===")
display(df_month_trends) if 'display' in globals() else print(df_month_trends.to_string(index=False))

=== SEASONAL BREAKDOWN OF TEMPERATURE TRENDS (DHAKA, 2001-2025) ===
    Month  Slope (°C/decade)  Total Change (°C)  p-value Significant (p<0.05)
  January              0.098               0.24   0.7437                   NO
 February             -0.305              -0.76   0.3589                   NO
    March             -0.235              -0.59   0.3695                   NO
    April             -0.155              -0.39   0.6339                   NO
      May             -0.353              -0.88   0.1274                   NO
     June             -0.024              -0.06   0.8602                   NO
     July              0.167               0.42   0.0207                  YES
   August              0.075               0.19   0.3305                   NO
September              0.361               0.90   0.0000                  YES
  October              0.452               1.13   0.0062                  YES
 November              0.195               0.49   0.3347                  

---
## 🏆 Conclusion & Roadmap to Phase 2 (Regional Grid Analysis)

### Key Findings from Phase 1:
1. **NASA Model Verification:** Successfully ingested 25 continuous years (9,131 daily records) of NASA GMAO MERRA-2 assimilation model data via NASA POWER API.
2. **Data Integrity:** 0 missing values, complete daily temporal continuity.
3. **Statistical Verdict on Annual Mean:** While Dhaka's annual mean surface temperature shows a slight positive trend ($+0.026$ °C/decade), it is **not statistically significant** at the $\alpha = 0.05$ level ($p = 0.757$). This proves why rigorous statistical testing is critical—preventing false claims of "drastic uniform warming" when data shows high inter-annual oscillation.
4. **Seasonal Masking Effect:** Individual months and seasons demonstrate divergent signals, illustrating how one aggregate variable can conceal significant seasonal extremes.

### Next Step (Phase 2):
Extend this exact scientific engine across the entire geographical bounding box of Bangladesh ($20.5^\circ\text{N} - 26.5^\circ\text{N}, 88.0^\circ\text{E} - 92.8^\circ\text{E}$) on a spatial grid to answer: **"WHERE is change happening, and is it statistically significant across different divisions (Sylhet vs. Rajshahi vs. Chittagong)?"**